# Section 6 — Encoding the First Tree: the Label Counters

This notebook tests the implementation of **Section 6**:
1. **Definition 5 (Winning threshold and label counters)**: Winning threshold $W = \lfloor D / 2 \rfloor + 1$, counter bit width $M + 1$ with $M = \lceil \log_2(D / 2) \rceil + 1$, and win flags $\text{label\_win}^\ell$.
2. **Definition 6 (Path formula)**: Encoding $\pi_b$ for each leaf as conjunction of positive ($\le$) and negative ($>$) node test comparators.
3. **Definition 7 (First-tree formula $\Phi_0$)**: Initializing the winning leaf's counter to 1, all rival counters to 0, and all win flags to $\text{False}$.

In [1]:
import os
import sys
import numpy as np
import dd.autoref as _bdd

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd.forest import Tree, Forest
from rfbdd.discretize import discretize
from rfbdd.first_tree import (
    compute_counter_width_m,
    counter_bit_name,
    win_flag_name,
    declare_counter_and_win_vars,
    encode_counter_equals,
    encode_path_formula,
    encode_first_tree,
)

def check(condition: bool, message: str) -> None:
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {message}")
    assert condition, message

print("Setup complete.")

Setup complete.


## 6.1 Sample Forest and Counter Parameters (Definition 5)

We use our test forest ($D = 2$ trees, $F = 2$ features, $|L| = 2$ classes):
- $W = \lfloor 2 / 2 \rfloor + 1 = 2$.
- $M = \lceil \log_2(2 / 2) \rceil + 1 = 0 + 1 = 1$.
- Counter width: $M + 1 = 2$ bits ($j \in \{0, 1\}$, representing values $0, 1, 2, 3$).

In [2]:
t0 = Tree(
    feature=np.array([0, -1, 1, -1, -1], dtype=np.int64),
    threshold=np.array([5.7, np.nan, 10.0, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 0, 1], dtype=np.int64),
)
t1 = Tree(
    feature=np.array([0, -1, 0, -1, -1], dtype=np.int64),
    threshold=np.array([2.5, np.nan, 8.1, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 1, 0], dtype=np.int64),
)
forest = Forest(trees=[t0, t1], n_features=2, n_classes=2, classes=np.array([0, 1]), name="toy_forest")
disc = discretize(forest)

check(forest.W == 2, f"Winning threshold W = {forest.W} (expected 2)")
M = compute_counter_width_m(forest.D)
check(M == 1, f"Counter parameter M = {M} (expected 1)")
print(f"Counter width is M + 1 = {M + 1} bits.")

[PASS] Winning threshold W = 2 (expected 2)
[PASS] Counter parameter M = 1 (expected 1)
Counter width is M + 1 = 2 bits.


## 6.2 Path Formulas $\pi_b$ (Definition 6)

In tree $T_0$, there are 3 leaves:
1. **Leaf 1** (class 0): path `[(0, feat=0, branch=1)]` $\implies x_0 \le 5.7$ (cell index $\le 1$).
2. **Leaf 3** (class 0): path `[(0, feat=0, branch=0), (2, feat=1, branch=1)]` $\implies x_0 > 5.7 \wedge x_1 \le 10.0$.
3. **Leaf 4** (class 1): path `[(0, feat=0, branch=0), (2, feat=1, branch=0)]` $\implies x_0 > 5.7 \wedge x_1 > 10.0$.

In [3]:
bdd = _bdd.BDD()
for var_name in disc.all_bit_names:
    bdd.declare(var_name)
declare_counter_and_win_vars(bdd, forest.n_classes, M)

# Check path mutually exclusive and partition of inputs
path_bdds = []
for steps, leaf_label in t0.paths():
    pi_b = encode_path_formula(bdd, disc, steps, disc.taus[0])
    path_bdds.append(pi_b)

# Paths are mutually exclusive:
check(path_bdds[0] & path_bdds[1] == bdd.false, "Paths 0 and 1 are disjoint")
check(path_bdds[0] & path_bdds[2] == bdd.false, "Paths 0 and 2 are disjoint")
check(path_bdds[1] & path_bdds[2] == bdd.false, "Paths 1 and 2 are disjoint")

# Paths cover the entire input space:
all_paths_union = path_bdds[0] | path_bdds[1] | path_bdds[2]
check(all_paths_union == bdd.true, "Union of paths covers entire input space (Top)")

[PASS] Paths 0 and 1 are disjoint
[PASS] Paths 0 and 2 are disjoint
[PASS] Paths 1 and 2 are disjoint
[PASS] Union of paths covers entire input space (Top)


## 6.3 Construction and Evaluation of $\Phi_0$ (Definition 7)

We build $\Phi_0$ and test it against concrete samples:
- Sample $A = [3.0, 5.0]$: feature $0$ is in cell $1 \le 1 \implies$ routes to Leaf 1 (class 0).
  Expected state: $LC^0 = 1$, $LC^1 = 0$, $\text{win}_0 = \text{False}$, $\text{win}_1 = \text{False}$.
- Sample $B = [7.0, 12.0]$: feature 0 is in cell $2 > 1$, feature 1 is in cell $1 > 0 \implies$ routes to Leaf 4 (class 1).
  Expected state: $LC^0 = 0$, $LC^1 = 1$, $\text{win}_0 = \text{False}$, $\text{win}_1 = \text{False}$.

In [4]:
phi_0 = encode_first_tree(bdd, forest, disc)
check(phi_0 != bdd.false, "Phi_0 is satisfiable")

# Sample A: [3.0, 5.0] -> cells [1, 0] -> predicts class 0
assign_A = disc.encode_sample(np.array([3.0, 5.0]))
state_A_correct = {
    "lc_0_0": True, "lc_0_1": False,  # LC^0 = 1
    "lc_1_0": False, "lc_1_1": False, # LC^1 = 0
    "win_0": False, "win_1": False,    # no winners
}
eval_A = bdd.let(assign_A | state_A_correct, phi_0)
check(eval_A == bdd.true, "Sample A satisfies Phi_0 with LC^0 = 1, LC^1 = 0")

# Wrong state for Sample A (LC^1 = 1) must be rejected
state_A_wrong = {
    "lc_0_0": False, "lc_0_1": False, # LC^0 = 0
    "lc_1_0": True, "lc_1_1": False,  # LC^1 = 1
    "win_0": False, "win_1": False,
}
eval_A_wrong = bdd.let(assign_A | state_A_wrong, phi_0)
check(eval_A_wrong == bdd.false, "Sample A rejects wrong counter state")

# Sample B: [7.0, 12.0] -> cells [2, 1] -> predicts class 1
assign_B = disc.encode_sample(np.array([7.0, 12.0]))
state_B_correct = {
    "lc_0_0": False, "lc_0_1": False, # LC^0 = 0
    "lc_1_0": True, "lc_1_1": False,  # LC^1 = 1
    "win_0": False, "win_1": False,
}
eval_B = bdd.let(assign_B | state_B_correct, phi_0)
check(eval_B == bdd.true, "Sample B satisfies Phi_0 with LC^0 = 0, LC^1 = 1")

[PASS] Phi_0 is satisfiable
[PASS] Sample A satisfies Phi_0 with LC^0 = 1, LC^1 = 0
[PASS] Sample A rejects wrong counter state
[PASS] Sample B satisfies Phi_0 with LC^0 = 0, LC^1 = 1
